# Combined Survival Analysis Evaluation

Comparing Standard CoxPH, DeepCoxPH, and DeepSurvivalMachines using standardized split.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from scipy.stats import f_oneway, chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
plt.rcParams["figure.dpi"] = 1000
plt.rcParams["savefig.dpi"] = 1000
# Survival Analysis Imports
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.util import Surv
from sksurv.metrics import (
    concordance_index_censored, 
    cumulative_dynamic_auc, 
    integrated_brier_score, 
    brier_score,
    concordance_index_ipcw
)
from auton_survival.models.cph import DeepCoxPH
from auton_survival.models.dsm import DeepSurvivalMachines

# Plotting settings
plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings("ignore")

In [2]:
# 1. Load Data
df = pd.read_csv("AIDS_Classification_50000.csv")
print(f"Dataset Shape: {df.shape}")

# Define columns
time_col = 'time'
event_col = 'infected'

X = df.drop(columns=[time_col, event_col])
t = df[time_col].values
e = df[event_col].values

cat_cols = [col for col in X.columns if X[col].nunique() < 10]
num_cols = [col for col in X.columns if col not in cat_cols]

# 2. Split Data (Standardized for all models)
# Using 70% Train, 15% Test, 15% Val
x_train_raw, x_temp, t_train, t_temp, e_train, e_temp = train_test_split(
    X, t, e, test_size=0.30, random_state=42, stratify=e
)
x_test_raw, x_val_raw, t_test, t_val, e_test, e_val = train_test_split(
    x_temp, t_temp, e_temp, test_size=0.30, random_state=42, stratify=e_temp
)

# Remove max values from test/val (often causes issues in evaluation if they exceed train max)
mask_test = t_test < t_train.max()
x_test_raw = x_test_raw[mask_test]
t_test = t_test[mask_test]
e_test = e_test[mask_test]

mask_val = t_val < t_train.max()
x_val_raw = x_val_raw[mask_val]
t_val = t_val[mask_val]
e_val = e_val[mask_val]

print(f"Train: {x_train_raw.shape}, Test: {x_test_raw.shape}, Val: {x_val_raw.shape}")

# 3. Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
    ],
    remainder="passthrough",
)

x_train = preprocessor.fit_transform(x_train_raw)
x_test = preprocessor.transform(x_test_raw)
x_val = preprocessor.transform(x_val_raw)

# Prepare sksurv format targets
y_train_surv = Surv.from_arrays(e_train.astype(bool), t_train)
y_test_surv = Surv.from_arrays(e_test.astype(bool), t_test)
y_val_surv = Surv.from_arrays(e_val.astype(bool), t_val)

# Data dictionary for Auton models
# (Auton models assume numpy arrays for inputs, handled above by fit_transform returning numpy/sparse)
# If sparse, densify for NN
if hasattr(x_train, 'toarray'):
    x_train = x_train.toarray()
    x_test = x_test.toarray()
    x_val = x_val.toarray()

Dataset Shape: (50000, 23)
Train: (35000, 21), Test: (10480, 21), Val: (4495, 21)


In [3]:
# Statistical Analysis
# Statistical Analysis of Data Splits
# Comparing Original, Train, Test, and Validation sets using ANOVA (Numerical) and Chi-Squared (Categorical)

stats_results = []

# Helper to get data for a column across splits
# Note: 'Original' is df (or X+t+e). 'Train' is x_train_raw + t_train + e_train...
# We will construct temporary Series for easier handling

# Combine features and targets for splits
df_train = x_train_raw.copy()
df_train[time_col] = t_train
df_train[event_col] = e_train

df_test = x_test_raw.copy()
df_test[time_col] = t_test
df_test[event_col] = e_test

df_val = x_val_raw.copy()
df_val[time_col] = t_val
df_val[event_col] = e_val

# List of all variables to check
all_vars = num_cols + cat_cols + [time_col, event_col]
# Determine type for time/event
type_map = {c: 'Numeric' for c in num_cols}
type_map.update({c: 'Categorical' for c in cat_cols})
type_map[time_col] = 'Numeric'
type_map[event_col] = 'Categorical'

print(f"{'Variable':<20} | {'Type':<12} | {'P-Value':<10} | {'Result':<10}")
print("-" * 60)

for var in all_vars:
    # Get values
    v_orig = df[var].values
    v_train = df_train[var].values
    v_test = df_test[var].values
    v_val = df_val[var].values
    
    var_type = type_map[var]
    p_val = np.nan
    
    if var_type == 'Numeric':
        # ANOVA
        stat, p_val = f_oneway(v_train, v_test, v_val)
        # Note: We compare the splits against each other to check for distribution shift.
        # Ideally, we want them to be from the same distribution (High p-value > 0.05).
        
    elif var_type == 'Categorical':
        # Chi-Squared
        # Create contingency table: [Values] x [SplitLabel]
        # But we need counts.
        # easier: get value counts for each split, align index (categories)
        
        # Get all unique categories
        cats = np.unique(np.concatenate([v_train, v_test, v_val]))
        
        counts_train = [np.sum(v_train == c) for c in cats]
        counts_test = [np.sum(v_test == c) for c in cats]
        counts_val = [np.sum(v_val == c) for c in cats]
        
        # Contingency table shape: (n_categories, 3_splits)
        obs = np.array([counts_train, counts_test, counts_val]).T
        
        # Filter out zero rows if any (shouldn't happen if initialized from union of cats)
        # Chi2 requires frequencies >= 5 usually, but we accept as is for large N
        chi2, p_val, dof, ex = chi2_contingency(obs)
        
    sig = "Different" if p_val < 0.05 else "Same"
    stats_results.append({
        "Variable": var,
        "Type": var_type,
        "P-Value": p_val,
        "Significance (<0.05)": sig
    })
    
    print(f"{var:<20} | {var_type:<12} | {p_val:.4f}     | {sig}")

df_stats = pd.DataFrame(stats_results)

Variable             | Type         | P-Value    | Result    
------------------------------------------------------------
age                  | Numeric      | 0.3523     | Same
wtkg                 | Numeric      | 0.3234     | Same
karnof               | Numeric      | 0.1120     | Same
preanti              | Numeric      | 0.6399     | Same
cd40                 | Numeric      | 0.3558     | Same
cd420                | Numeric      | 0.9305     | Same
cd80                 | Numeric      | 0.5571     | Same
cd820                | Numeric      | 0.3752     | Same
trt                  | Categorical  | 0.6734     | Same
hemo                 | Categorical  | 0.0921     | Same
homo                 | Categorical  | 0.9499     | Same
drugs                | Categorical  | 0.7912     | Same
oprior               | Categorical  | 0.2312     | Same
z30                  | Categorical  | 0.7864     | Same
race                 | Categorical  | 0.9856     | Same
gender               | Categorical  |

In [4]:
# Compute Time Quartiles for Evaluation
horizons = [0.25, 0.50, 0.75]
times = np.quantile(t[e == 1], horizons).tolist()
# Filter 'times' to ensure they are within the test range and valid
times = [t_v for t_v in times if t_v < t_train.max() and t_v > t_train.min()]
print(f"Evaluation Time Horizons (Quartiles): {times}")

Evaluation Time Horizons (Quartiles): [496.0, 992.0, 1127.0]


In [6]:
models = {}

# --- 1. Standard CoxPH (scikit-survival) ---
print("Training Standard CoxPH...")
# Parameters from rsf_coxph.ipynb: alpha=21.544
cph = CoxPHSurvivalAnalysis(alpha=21.54434690031882)
cph.fit(x_train, y_train_surv)
models['CoxPH'] = cph

# --- 2. Deep Survival Machines (auton-survival) ---
print("Training Deep Survival Machines...")
# Parameters from deep_survival_machines.ipynb: distribution='LogNormal', k=4, layers=[100], lr=1e-5
dsm = DeepSurvivalMachines(distribution='LogNormal', 
        k=4, 
        layers=[100])
dsm.fit(x_train, t_train, e_train, val_data=(x_val, t_val, e_val), 
        learning_rate=1e-5, iters=100) # batch_size=64)
models['DSM'] = dsm

# --- 3. Deep CoxPH (auton-survival) ---
print("Training Deep CoxPH...")
# Parameters from auton_cph.ipynb: layers=[64, 64], lr=1e-4, optimizer='RMSProp'
dcph = DeepCoxPH(layers=[64, 64])
dcph.fit(x_train, t_train, e_train, val_data=(x_val, t_val, e_val), 
         learning_rate=1e-4, optimizer='RMSProp', iters=100) # batch_size=64)
models['DeepCoxPH'] = dcph

print("All models trained.")

Training Standard CoxPH...
Training Deep Survival Machines...


100%|██████████| 100/100 [01:15<00:00,  1.33it/s]


Training Deep CoxPH...


 37%|███▋      | 37/100 [00:13<00:22,  2.80it/s]

All models trained.


In [7]:
# Define the Bootstrap Helper Function
def bootstrap_auc_ci(y_train_surv, y_test_surv, risk_at_times, times, n_boot=100, random_seed=42):
    """Calculate 95% Confidence Intervals for AUC at each horizon and Integrated AUC using bootstrapping."""
    np.random.seed(random_seed)
    boot_quartile_aucs = []
    boot_integrated_aucs = []
    
    n_test = len(y_test_surv)
    indices = np.arange(n_test)
    
    for _ in range(n_boot):
        # Resample test set indices with replacement
        boot_idx = np.random.choice(indices, size=n_test, replace=True)
        
        # Calculate AUC for resampled test set
        try:
            aucs, i_auc = cumulative_dynamic_auc(y_train_surv, y_test_surv[boot_idx], risk_at_times[boot_idx], times)
            boot_quartile_aucs.append(aucs)
            boot_integrated_aucs.append(i_auc)
        except:
            # Skip iterations where bootstrapping might result in no events
            continue
            
    boot_quartile_aucs = np.array(boot_quartile_aucs)
    boot_integrated_aucs = np.array(boot_integrated_aucs)
    
    # Calculate 95% CI (2.5th and 97.5th percentiles)
    q_lower = np.percentile(boot_quartile_aucs, 2.5, axis=0)
    q_upper = np.percentile(boot_quartile_aucs, 97.5, axis=0)
    
    i_lower = np.percentile(boot_integrated_aucs, 2.5)
    i_upper = np.percentile(boot_integrated_aucs, 97.5)
    
    return q_lower, q_upper, i_lower, i_upper

In [8]:
results_quartile = []
results_global = []

for name, model in models.items():
    print(f"Evaluating {name}...")
    
    # 1. Predict Risk / Survival
    # Different APIs handling
    if name in ['CoxPH', 'RSF'] :
        # sksurv
        risk_scores = model.predict(x_test)
        surv_funcs = model.predict_survival_function(x_test)
        # Get probability of survival at specific 'times'
        surv_probs = np.array([[fn(t) for t in times] for fn in surv_funcs])
        risk_at_times = 1 - surv_probs # Cumulative incidence approximation for AUC
        
        # For global C-index, use the single risk score
        global_risk = risk_scores
        
    else:
        # auton-survival (DeepCoxPH, DSM)
        risk_at_times = model.predict_risk(x_test, t=times)
        surv_probs = model.predict_survival(x_test, t=times)
        
        # For global C-index, approximate global risk (e.g. mean risk over horizons or expected survival)
        # Using mean risk over the 3 horizons for simplicity and consistency with previous notes
        global_risk = np.mean(risk_at_times, axis=1)
     # --- Bootstrapping for AUC CI ---
    print(f"Calculating Bootstrap 95% CI for {name} AUC (n=500)...")
    q_lower, q_upper, i_lower, i_upper = bootstrap_auc_ci(y_train_surv, y_test_surv, risk_at_times, times, n_boot=500)

    # --- Quartile Metrics ---
    for i, t_val in enumerate(times):
        # Time-dependent C-index
        # CoxPH has proportional hazards -> single risk score valid for all times
        # But specific SkSurv c-index expects one risk score.
        # Ideally use the risk_at_time for TD-Cindex if available, or the global risk.
        # Using global risk is standard for PH models, but let's use the time-specific risk surface if we calculated it (1-Surv).

        
        current_risk = risk_at_times[:, i]

        ctd = concordance_index_ipcw(y_train_surv, y_test_surv, current_risk, t_val)[0]
        
        # Brier Score
        # brier_score returns (times, scores), we want the score at index i
        # But we computed surv_probs[:, i] manually for Cox.
        # Let's use the bulk brier function for consistency
        # For specific time t_val:
        bs = brier_score(y_train_surv, y_test_surv, surv_probs[:, i].reshape(-1, 1), [t_val])[1][0]
        
        # AUC
        # Cumulative Dynamic AUC returns (aucs, mean_auc)
        # We need the AUC at this specific time t_val.
        # We can pass just this one time point.
        auc_val_tuple = cumulative_dynamic_auc(y_train_surv, y_test_surv, current_risk, t_val)
        auc = auc_val_tuple[0][0]
        
        results_quartile.append({
            "Model": name,
            "Quartile_Time": t_val,
            "Quartile_Idx": f"Q{i+1}",
            "C-Index (TD)": ctd,
            "Brier Score": bs,
            "AUC Score": auc,
            "AUC_95%_CI": f"({q_lower[i]:.4f}, {q_upper[i]:.4f})"
        })

    # --- Global Metrics ---
    # Integrated Brier / AUC
    ibs = integrated_brier_score(y_train_surv, y_test_surv, surv_probs, times)
    
    # Integrated AUC
    _, i_auc = cumulative_dynamic_auc(y_train_surv, y_test_surv, risk_at_times, times)
    
    # Global Concordance (Traditional C-index for censored data)
    # Note: For time-varying risk models, this is an approximation using a representative risk score
    c_global = concordance_index_censored(y_test_surv["event"], y_test_surv["time"], global_risk)[0]
    
    results_global.append({
        "Model": name,
        "Integrated Brier Score": ibs,
        "Integrated ROC-AUC": i_auc,
        "AUC_Integrated_95%_CI": f"({i_lower:.4f} - {i_upper:.4f})",
        "Global C-Index": c_global
    })

df_quartile = pd.DataFrame(results_quartile)
df_global = pd.DataFrame(results_global)

Evaluating CoxPH...
Calculating Bootstrap 95% CI for CoxPH AUC (n=500)...
Evaluating DSM...
Calculating Bootstrap 95% CI for DSM AUC (n=500)...
Evaluating DeepCoxPH...
Calculating Bootstrap 95% CI for DeepCoxPH AUC (n=500)...


In [9]:
print("--- Metrics by Quartile ---")
display(df_quartile)

print("\n--- Global Metrics (Without Quartiles) ---")
display(df_global)

--- Metrics by Quartile ---


,Model,Quartile_Time,Quartile_Idx,C-Index (TD),Brier Score,AUC Score,AUC_95%_CI
0,CoxPH,496.0,Q1,0.698695,0.070929,0.706847,"(0.6889, 0.7249)"
1,CoxPH,992.0,Q2,0.684666,0.139520,0.697676,"(0.6838, 0.7106)"
2,CoxPH,1127.0,Q3,0.657435,0.205832,0.657372,"(0.6432, 0.6716)"
3,DSM,496.0,Q1,0.698413,0.071619,0.706696,"(0.6888, 0.7236)"
4,DSM,992.0,Q2,0.683769,0.153860,0.695968,"(0.6825, 0.7096)"
5,DSM,1127.0,Q3,0.653429,0.211899,0.647879,"(0.6327, 0.6617)"
6,DeepCoxPH,496.0,Q1,0.704615,0.070797,0.713146,"(0.6965, 0.7309)"
7,DeepCoxPH,992.0,Q2,0.686283,0.139191,0.700018,"(0.6862, 0.7133)"
8,DeepCoxPH,1127.0,Q3,0.660326,0.204535,0.663018,"(0.6483, 0.6771)"



--- Global Metrics (Without Quartiles) ---


,Model,Integrated Brier Score,Integrated ROC-AUC,AUC_Integrated_95%_CI,Global C-Index
0,CoxPH,0.119656,0.682507,(0.6711 - 0.6948),0.670418
1,DSM,0.127745,0.677815,(0.6664 - 0.6897),0.668616
2,DeepCoxPH,0.119300,0.687260,(0.6757 - 0.6997),0.672772


In [17]:
from sklearn.metrics import roc_curve, auc

# Plotting ROC-AUC for each quartile
fig, axes = plt.subplots(1, 3, figsize=(30, 10), sharey=True)

for i, t_val in enumerate(times):
    ax = axes[i]
    ax.set_title(f"ROC Curve at Q{i+1} (t={t_val:.1f})", weight = "bold")
    
    for name, model in models.items():
        # Get Risk Scores for this time
        if name in ['CoxPH', 'RSF']:
            surv_funcs = model.predict_survival_function(x_test)
            surv_prob_t = np.array([fn(t_val) for fn in surv_funcs])
            risk_t = 1 - surv_prob_t
        else:
            risk_t = model.predict_risk(x_test, t=t_val).flatten()
            
        # Simplified Approach for Visualization:
        # Define 'label' as: 1 if event happened <= t, 0 if event > t. 
        # Exclude censored subjects before t (who are ambiguous).
        
        mask = ~((e_test == 0) & (t_test < t_val)) # Keep: (Event <= t) OR (Censored > t) OR (Event > t)
        
        ids = np.where(mask)[0]
        y_binary = (e_test[ids] == 1) & (t_test[ids] <= t_val)
        y_score = risk_t[ids]
        
        if len(np.unique(y_binary)) < 2:
            print(f"Skipping plot for {name} at {t_val}: Not enough classes.")
            continue
            
        fpr, tpr, _ = roc_curve(y_binary.astype(int), y_score)
        roc_auc_val = auc(fpr, tpr)
        CI = df_global.loc[df_global["Model"] == name]["AUC_Integrated_95%_CI"].values[0]
        ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_val:.4f} CI={CI})", linewidth=2)
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=1)
    ax.set_xlabel("False Positive Rate", weight = "bold")
    ax.set_ylabel("True Positive Rate", weight = "bold")
    ax.legend()
    
plt.tight_layout()
plt.savefig('roc_curve_quartiles.png')
plt.show()    

In [18]:
display(df_quartile.T)
df_quartile.T.to_csv('results_quartile.csv')
display(df_global.T)
df_global.T.to_csv('results_global.csv')

,0,1,2,3,4,5,6,7,8
Model,CoxPH,CoxPH,CoxPH,DSM,DSM,DSM,DeepCoxPH,DeepCoxPH,DeepCoxPH
Quartile_Time,496.0,992.0,1127.0,496.0,992.0,1127.0,496.0,992.0,1127.0
Quartile_Idx,Q1,Q2,Q3,Q1,Q2,Q3,Q1,Q2,Q3
C-Index (TD),0.698695,0.684666,0.657435,0.698413,0.683769,0.653429,0.704615,0.686283,0.660326
Brier Score,0.070929,0.13952,0.205832,0.071619,0.15386,0.211899,0.070797,0.139191,0.204535
AUC Score,0.706847,0.697676,0.657372,0.706696,0.695968,0.647879,0.713146,0.700018,0.663018
AUC_95%_CI,"(0.6889, 0.7249)","(0.6838, 0.7106)","(0.6432, 0.6716)","(0.6888, 0.7236)","(0.6825, 0.7096)","(0.6327, 0.6617)","(0.6965, 0.7309)","(0.6862, 0.7133)","(0.6483, 0.6771)"


,0,1,2
Model,CoxPH,DSM,DeepCoxPH
Integrated Brier Score,0.119656,0.127745,0.1193
Integrated ROC-AUC,0.682507,0.677815,0.68726
AUC_Integrated_95%_CI,(0.6711 - 0.6948),(0.6664 - 0.6897),(0.6757 - 0.6997)
Global C-Index,0.670418,0.668616,0.672772
